# Epsilon Fund - Walk-Forward Validation
Uses `infrastructure/walk_forward/` to run rolling Optuna optimisation and evaluate OOS robustness. Ideal to use after finding strategy that seems to work using backtesting framework to ensure logic is valid.

---
### Iteration Guidelines

**Overfitting the iteration process:** Each time you inspect OOS results and adjust parameters, you leak OOS information into your design decisions. Cap yourself at **3–4 iterations** — first run with everything free, second with obvious fixes from CV + plateau analysis, third to tighten remaining params. 

If the strategy still shows heavy overfitting signals after three passes, **the problem is the strategy architecture, not the parameters**.

**WFE:** Walk-forward efficiency - examine IS/OOS ratio (simplest).

**Pertubation degradation:** Examine pertubation table to see if degradation reduces across runs.

| Signal | Meaning | Action |
|--------|---------|--------|
| IS Sharpe drops, OOS Sharpe holds or rises, WFE improves | Removing noise-fitting degrees of freedom | Continue iterating |
| Perturbation degradation shrinks across iterations | Parameters becoming more robust | Continue iterating |
| N/A plateau params decreasing across iterations | Strategy becoming more tolerant of parameter movement | Continue iterating |
| WFE improvement flattens (e.g. 0.55 → 0.65 → 0.67) | Diminishing returns — further fixes won't help much | Stop iterating |
| IS and OOS both decline but WFE rises (IS falls faster) | Constraining away real signal, not just noise | Stop iterating |
| OOS Sharpe keeps declining despite "better" param setup | Overfitting the iteration process itself | Stop — problem is strategy architecture, not parameters |
| WFE decreases after fixing a parameter | Locked in a param that was legitimately adapting across folds | Unfix that parameter and re-run |

---

In [56]:
import sys
import os
import pandas as pd
import numpy as np


# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows path
# ─────────────────────────────────────────────────────────────────────────────

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))


from binance_client import get_binance_client, get_data
from wf_engine import walk_forward, plateau_analysis, plateau_summary, perturbation_test, cost_stress_test
from wf_visualizer import plot_walk_forward_results, plot_plateau_analysis
from engine import backtest


---
## Data

**Pairs** — any Binance pair in `BASEQUOTE` format (e.g. `BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `BNBUSDT`).  
Verify availability at [binance.com/en/trade](https://www.binance.com/en/trade).

**Intervals** — `'1m'` `'5m'` `'15m'` `'1h'` `'4h'` `'1d'` `'1w'`

**Lookback** — days of history: must be >= (train_bars + test_bars) * n_folds desired

In [57]:
SYMBOL   = 'ETHUSDT'
INTERVAL = '1d'
LOOKBACK = 2150 


# ── Multiple pairs (optional) ──────────────────────────────────────────────────
# SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
# data_dict = get_multiple_data(client, SYMBOLS, INTERVAL, LOOKBACK)
# Access via: data_dict['BTCUSDT_1d'], data_dict['ETHUSDT_1d'] ...
# ──────────────────────────────────────────────────────────────────────────────

client   = get_binance_client()
df = get_data(client, SYMBOL, INTERVAL, LOOKBACK)
print(f'Data: {df.index[0].date()} → {df.index[-1].date()}  ({len(df)} bars)')
df.tail(3)

Data: 2020-05-24 → 2026-04-12  (2150 bars)


,Open,High,Low,Close,Volume
Time,,,,,
2026-04-10,2190.17,2257.99,2176.10,2245.05,272899.0858
2026-04-11,2245.06,2329.93,2229.00,2284.99,214785.4039
2026-04-12,2285.00,2288.57,2207.06,2216.10,130517.2557


---
## Parameter Configuration

Define which parameters to optimise and anchor - **recommended to do after strategy writeup**

`FIXED_PARAMS`: choose parameters with CV < 0.15 from prior stability run, cross referencing with pertubation verdict table to reduce search space, improve OOS credibility.

**Practical rule**: free parameter count to be **at most** `n_trials` / 20 for meaningful conversion. 

> e.g 400 trials: ~20 free params as the theoretical ceiling, in practice you want far fewer because TPE (Optuna method) efficiency degrades exponentially with dimensionality, not linearly. A good target for 400 trials is 6–10 free parameters.

 'ema_span': 28,
    'adx_override': 61,
    'risk_per_trade': 0.045,x
    'max_leverage': 2.5494,x
    'stop_mult_pos_caution': 0.4935,
    'stop_mult_pos_normal': 1.676,
    'stop_atr_scale': 1.8,
    'obv_lookback': 27, x
    'atr_stop': 27,x
    'obv_ma_period': 21,
    'vol_ma_period': 38


In [125]:
# ── parameter search space ────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
# Only keys present in PARAMS above are searched — remove a key from PARAMS to exclude it entirely.

PARAM_DEFS = {
    'ema_span':          ('int',   5,    40),
    'swing_caution':     ('int',   3,    14),
    'swing_stop':        ('int',   3,    10),
    'atr_caution':       ('int',   10,   30),
    'atr_stop':          ('int',   24,   30),
    'atr_size':          ('int',   3,    14),
    'adx_override':      ('int',   40,   80),
    'stop_atr_scale':    ('float', 0.5,  2.0),
    'risk_per_trade':    ('float', 0.005, 0.05),
    'max_leverage':      ('float', 1.0,  3.0),
    'stop_mult_pos_caution': ('float', 0.1, 0.9),
    'stop_mult_pos_normal':  ('float', 0.8, 2.0),
    'stop_mult_ent_both':    ('float', 0.5, 2.5),
    'stop_mult_ent_caution': ('float', 0.1, 0.9),
    'stop_mult_ent_normal':  ('float', 0.5, 1.5),
    'vol_ma_period':  ('int',   35,  40),   # rolling window for volume MA
    'obv_ma_period':  ('int',   18,  24),   # OBV smoothing window
    'obv_lookback':   ('int',   24,  30),   # bars back to compare price vs OBV direction
}

# ── anchored params (set after a stability run; empty first time) ─────────────
# These bypass Optuna and are held constant across all folds.
# Populate using stability_df results: fix params with CV < 0.15
FIXED_PARAMS = {
    'risk_per_trade': 0.045,
    'max_leverage': 2.6,

    'ema_span': 28,

 #   'obv_lookback': 19

  #  'stop_mult_pos_caution': 0.6298,
    
   # 'adx_override': 61,

    'stop_mult_pos_caution': 0.5,
 #   'stop_mult_pos_normal': 1.676,
 #   'stop_atr_scale': 1.8,

    'obv_lookback': 28, 
   # 'atr_stop': 27,
    'obv_ma_period': 20,
  #  'vol_ma_period': 38,

    'swing_caution': 5,
  #  'stop_atr_scale': 1.404,


 
    }



### *Guide to parameter anchoring*

|  | Robust Plateau| Fragile Plateau |
|--|-------------------|-------------------|
| Low CV | Stable across folds and insensitive to exact value - keep free| Looks stable but is fitting the same noise patterns - fix at concensus|
| High CV | Parameter unimportant - fix at any reasonable value | Unstable across folds and sitting on a cliff - strong candidate to eliminate |

Copy-paste plateau analysis table above into fixed params section and decide manually on which to fix/keep free.c

---
## Strategy

**CRUCIAL** - Strategy logic needs to work well in backtesting notebook before running here, saves time not running walk-forward for a broken strategy.

**Available columns:** `Open` `High` `Low` `Close` `Volume`

**Required output:** a `position` column — `1` long · `0` flat · `-1` short  
**Optional:** `position_size` column (0–1) to use fractional capital

> The engine applies a 1-bar execution lag automatically. Inside the strategy loop, use `prev` for signal decisions and `curr` for execution — no manual shifting needed.

**To implement your strategy:**
1. Write strategy logic — compute indicators, signals, and execution loop: use `param_name`for those to be searched
2. Update `indicator_cols` to list your longest-warmup indicators — the engine uses this to clean NaN rows after OOS trimming


In [59]:
def my_strategy(df_slice: pd.DataFrame, params: dict) -> pd.DataFrame:

    df = df_slice.copy()

    # ── indicators ────────────────────────────────────────────────────────────
    df['EMA']          = df['Close'].ewm(span=params['ema_span'], adjust=False).mean()
    df['Swing_Hi_Cau'] = df['High'].rolling(params['swing_caution']).max()
    df['Swing_Lo_Cau'] = df['Low'].rolling(params['swing_caution']).min()
    df['Swing_Hi_Stp'] = df['High'].rolling(params['swing_stop']).max()

    def atr(period):
        hl = df['High'] - df['Low']
        hc = (df['High'] - df['Close'].shift(1)).abs()
        lc = (df['Low']  - df['Close'].shift(1)).abs()
        return pd.concat([hl, hc, lc], axis=1).max(axis=1).ewm(span=period, adjust=False).mean()

    df['ATR_Cau'] = atr(params['atr_caution'])
    df['ATR_Stp'] = atr(params['atr_stop'])
    df['ATR_Sz']  = atr(params['atr_size'])

    up    = df['High'].diff();  down = -df['Low'].diff()
    pdm   = up.where((up > down) & (up > 0), 0.0)
    ndm   = down.where((down > up) & (down > 0), 0.0)
    atr14 = atr(14)
    pdi   = 100 * pdm.ewm(span=14, adjust=False).mean() / atr14
    ndi   = 100 * ndm.ewm(span=14, adjust=False).mean() / atr14
    dx    = (100 * (pdi - ndi).abs() / (pdi + ndi).replace(0, np.nan)).fillna(0)
    df['ADX_14'] = dx.ewm(span=14, adjust=False).mean()

    df['Vol_MA']  = df['Volume'].rolling(params['vol_ma_period']).mean()
    direction     = df['Close'].diff().apply(lambda x: 1 if x > 0 else -1)
    df['OBV']     = (df['Volume'] * direction).cumsum()
    df['OBV_MA']  = df['OBV'].rolling(params['obv_ma_period']).mean()

    df['Caution_OBV']   = (df['Close'] > df['Close'].shift(params['obv_lookback'])) & (df['OBV'] < df['OBV_MA'])
    df['Caution_Long']  = ((df['Swing_Hi_Cau'] - df['Low']) > 1.5 * df['ATR_Cau']) | df['Caution_OBV']
    df['Caution_Short'] = ((df['High'] - df['Swing_Lo_Cau']) > 1.5 * df['ATR_Cau']) | (df['Close'] > df['EMA'])
    _valid = df['Swing_Hi_Stp'].notna() & df['ATR_Stp'].notna() & df['ATR_Sz'].notna() & df['OBV_MA'].notna() & df['Vol_MA'].notna()
    df['Entry_Long']    = (df['Close'] > df['EMA']) & (~df['Caution_Long'] | (df['ADX_14'] > params['adx_override'])) & (df['Volume'] > df['Vol_MA']) & _valid
    df['position_size_raw'] = (params['risk_per_trade'] / (df['ATR_Sz'] / df['Close'])).clip(0.1, params['max_leverage'])

    n             = len(df)
    position      = [0]      * n
    position_size = [0.0]    * n
    stop_arr      = [np.nan] * n
    in_position   = 0
    stop_loss     = np.nan
    current_size  = 0.0

    for i in range(1, n):
        prev = df.iloc[i - 1]
        curr = df.iloc[i]

        if in_position == 0:
            if curr['Entry_Long']:
                in_position  = 1
                current_size = curr['position_size_raw']
                cl = curr['Caution_Long']; cs = curr['Caution_Short']
                if cl and cs: sm = params['stop_mult_ent_both']
                elif cl:      sm = params['stop_mult_ent_caution']
                else:         sm = params['stop_mult_ent_normal']
                stop_loss = curr['Swing_Hi_Stp'] - curr['ATR_Stp'] * sm * params['stop_atr_scale']
        else:
            if prev['Close'] < stop_loss:
                in_position  = 0
                current_size = 0.0
                stop_loss    = np.nan
            else:
                sm        = params['stop_mult_pos_caution'] if curr['Caution_Long'] else params['stop_mult_pos_normal']
                stop_loss = max(stop_loss, curr['Swing_Hi_Stp'] - curr['ATR_Stp'] * sm * params['stop_atr_scale'])

        position[i]      = in_position
        position_size[i] = current_size
        stop_arr[i]      = stop_loss

    df['position']      = position
    df['position_size'] = position_size
    df['stop_loss']     = stop_arr

    indicator_cols = ['EMA', 'ATR_Cau', 'ADX_14', 'Swing_Hi_Cau', 'Vol_MA', 'OBV_MA']
    df['position']      = df['position'].fillna(0).astype(int)
    df['position_size'] = df['position_size'].fillna(0.0)
    df['stop_loss']     = df['stop_loss'].fillna(0.0)

    return df, indicator_cols


---
## Run Walk-Forward
Simulates how the strategy would have performed if re-optimised periodically
in live trading, and exposes whether good IS performance survives unseen data.

**Folds Setup**
| Parameter | Description | Guidance |
|---|---|---|
| `TRAIN_BARS` | Bars per training window | Aim for 2:1 to 3:1 ratio vs `TEST_BARS` |
| `TEST_BARS` | Bars per test window | `365` = ~1 year on daily data |
| `BURNIN_BARS` | Bars prepended to test for indicator warmup | Match your longest indicator period |
| `N_TRIALS` | Optuna trials per fold | 300–500 for daily; more = better but slower.10-20 trials per free parameter to avoid overfit |
| `COST` | Round-trip cost per trade | `0.001` = 0.1% |

Use `N_TRIALS` as robustness dia: if OOS degrades sharply as you increase it from 100→200→300, direct signal your parameter space still has too many degrees of freedom relative to the information content of the training window (consider decreasing). 

**Score and Rejection** — use to calibrate what Optuna optimises IS: default `score_fn(m)` uses weighted basket of Sharpe, Calmar and Return, normalised using their "max" value; default `reject_fn(m)` discards runs failing certain criteria that limits credibility.

> Pay attention to the **degradation ratio** — OOS/IS Sharpe reveals overfitting.

In [126]:
# ── walk-forward windows ──────────────────────────────────────────────────────
TRAIN_BARS  = 1050  
TEST_BARS   = 137   
BURNIN_BARS = 100   
N_TRIALS    = 400  
COST        = 0.001 

# ── SCORING FUNCTION ──────────────────────────────────────────────────────────
# Modify weights or swap components. Must return a float (higher = better).

def score_fn(m):
    SHARPE_MAX = 2.5
    CALMAR_MAX = 8
    RETURN_MAX = 9.0 #Calibrate to max of IS periods

    calmar = m['total_return'] / abs(m['max_drawdown']) if m['max_drawdown'] != 0 else 0

    s = np.clip(m['sharpe_ratio']  / SHARPE_MAX, 0, 1)
    c = np.clip(calmar             / CALMAR_MAX, 0, 1)
    r = np.clip(m['total_return']  / RETURN_MAX, 0, 1)

    return 0.50 * s + 0.30 * c + 0.20 * r

# ── REJECTION CRITERIA ────────────────────────────────────────────────────────
# Trials that return True are discarded (score → -999).

def reject_fn(m):
    if m is None:                      return True
    if m['num_trades']    < 15:        return True
    if m['win_rate']      < 0.3:      return True
    if m['max_drawdown']  < -0.7:     return True
    if m['profit_factor'] < 0.6:       return True
    return False


results = walk_forward(
    df           = df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    train_bars   = TRAIN_BARS,
    test_bars    = TEST_BARS,
    burnin_bars  = BURNIN_BARS,
    n_trials     = N_TRIALS,
    cost         = COST,
    score_fn     = score_fn,    # ← your notebook definition
    reject_fn    = reject_fn,   # ← your notebook definition
    save_csv     = None,
)

UPDATED WALK_FORWARD FILE IS RUNNING
Walk-forward: 8 fold(s)  train=1050  test=137  burnin=100  trials=400
  Fold 1: train 2020-05-24 → 2023-04-08  | test 2023-04-09 → 2023-08-23
  Fold 2: train 2020-10-08 → 2023-08-23  | test 2023-08-24 → 2024-01-07
  Fold 3: train 2021-02-22 → 2024-01-07  | test 2024-01-08 → 2024-05-23
  Fold 4: train 2021-07-09 → 2024-05-23  | test 2024-05-24 → 2024-10-07
  Fold 5: train 2021-11-23 → 2024-10-07  | test 2024-10-08 → 2025-02-21
  Fold 6: train 2022-04-09 → 2025-02-21  | test 2025-02-22 → 2025-07-08
  Fold 7: train 2022-08-24 → 2025-07-08  | test 2025-07-09 → 2025-11-22
  Fold 8: train 2023-01-08 → 2025-11-22  | test 2025-11-23 → 2026-04-08

Fixed (7): ['risk_per_trade', 'max_leverage', 'ema_span', 'stop_mult_pos_caution', 'obv_lookback', 'obv_ma_period', 'swing_caution']
Free  (11): ['swing_stop', 'atr_caution', 'atr_stop', 'atr_size', 'adx_override', 'stop_atr_scale', 'stop_mult_pos_normal', 'stop_mult_ent_both', 'stop_mult_ent_caution', 'stop_mult_e

  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 2.20  Return: 519.53%  DD: -26.10%  Calmar: 3.39  Trades: 65
  OOS → Sharpe: 0.43  Return: 2.05%  DD: -4.77%  Calmar: 1.17  Trades: 2

  Best params: {'risk_per_trade': 0.045, 'max_leverage': 2.6, 'ema_span': 28, 'stop_mult_pos_caution': 0.5, 'obv_lookback': 28, 'obv_ma_period': 20, 'swing_caution': 5, 'swing_stop': 3, 'atr_caution': 19, 'atr_stop': 30, 'atr_size': 10, 'adx_override': 41, 'stop_atr_scale': 1.4391795709202284, 'stop_mult_pos_normal': 1.7745727300587313, 'stop_mult_ent_both': 2.433293927051092, 'stop_mult_ent_caution': 0.26368179658736407, 'stop_mult_ent_normal': 0.5892839022860079, 'vol_ma_period': 36}

────────────────────────────────────────────────────────────
Fold 2/8  train: 2020-10-08 → 2023-08-23  test: 2023-08-24 → 2024-01-07


  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 2.43  Return: 541.10%  DD: -15.69%  Calmar: 5.78  Trades: 54
  OOS → Sharpe: 2.69  Return: 54.57%  DD: -9.19%  Calmar: 23.84  Trades: 13

  Best params: {'risk_per_trade': 0.045, 'max_leverage': 2.6, 'ema_span': 28, 'stop_mult_pos_caution': 0.5, 'obv_lookback': 28, 'obv_ma_period': 20, 'swing_caution': 5, 'swing_stop': 3, 'atr_caution': 23, 'atr_stop': 28, 'atr_size': 12, 'adx_override': 41, 'stop_atr_scale': 1.3671357308777885, 'stop_mult_pos_normal': 1.480665737184003, 'stop_mult_ent_both': 2.4850347023582193, 'stop_mult_ent_caution': 0.5012621087387751, 'stop_mult_ent_normal': 0.6214952888164831, 'vol_ma_period': 37}

────────────────────────────────────────────────────────────
Fold 3/8  train: 2021-02-22 → 2024-01-07  test: 2024-01-08 → 2024-05-23


...


  IS  → Sharpe: 2.12  Return: 519.01%  DD: -26.26%  Calmar: 3.37  Trades: 52
  OOS → Sharpe: 1.53  Return: 19.41%  DD: -13.79%  Calmar: 4.38  Trades: 5

  Best params: {'risk_per_trade': 0.045, 'max_leverage': 2.6, 'ema_span': 28, 'stop_mult_pos_caution': 0.5, 'obv_lookback': 28, 'obv_ma_period': 20, 'swing_caution': 5, 'swing_stop': 3, 'atr_caution': 21, 'atr_stop': 25, 'atr_size': 11, 'adx_override': 43, 'stop_atr_scale': 1.4266382502346153, 'stop_mult_pos_normal': 1.2797226404695103, 'stop_mult_ent_both': 1.4771272934297188, 'stop_mult_ent_caution': 0.6749173815453338, 'stop_mult_ent_normal': 0.6021019881648171, 'vol_ma_period': 37}

────────────────────────────────────────────────────────────
Fold 7/8  train: 2022-08-24 → 2025-07-08  test: 2025-07-09 → 2025-11-22


  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 2.08  Return: 524.01%  DD: -25.33%  Calmar: 3.51  Trades: 51
  OOS → Sharpe: 2.71  Return: 36.60%  DD: -11.77%  Calmar: 11.01  Trades: 7

  Best params: {'risk_per_trade': 0.045, 'max_leverage': 2.6, 'ema_span': 28, 'stop_mult_pos_caution': 0.5, 'obv_lookback': 28, 'obv_ma_period': 20, 'swing_caution': 5, 'swing_stop': 3, 'atr_caution': 21, 'atr_stop': 30, 'atr_size': 7, 'adx_override': 42, 'stop_atr_scale': 1.3098923518009524, 'stop_mult_pos_normal': 1.6717330952403238, 'stop_mult_ent_both': 1.726598147965634, 'stop_mult_ent_caution': 0.5280136638234875, 'stop_mult_ent_normal': 1.2511503049231694, 'vol_ma_period': 38}

────────────────────────────────────────────────────────────
Fold 8/8  train: 2023-01-08 → 2025-11-22  test: 2025-11-23 → 2026-04-08


  0%|          | 0/400 [00:00<?, ?it/s]


  IS  → Sharpe: 2.13  Return: 496.94%  DD: -22.18%  Calmar: 3.88  Trades: 42
  OOS → Sharpe: -0.56  Return: -4.19%  DD: -12.55%  Calmar: -0.86  Trades: 2

  Best params: {'risk_per_trade': 0.045, 'max_leverage': 2.6, 'ema_span': 28, 'stop_mult_pos_caution': 0.5, 'obv_lookback': 28, 'obv_ma_period': 20, 'swing_caution': 5, 'swing_stop': 4, 'atr_caution': 22, 'atr_stop': 28, 'atr_size': 14, 'adx_override': 71, 'stop_atr_scale': 1.113738267129752, 'stop_mult_pos_normal': 1.655340570450127, 'stop_mult_ent_both': 1.3416106012776279, 'stop_mult_ent_caution': 0.22687359217731445, 'stop_mult_ent_normal': 0.7398371407174732, 'vol_ma_period': 40}

════════════════════════════════════════════════════════════
WALK-FORWARD SUMMARY
════════════════════════════════════════════════════════════

Out-of-sample across 8 fold(s):
  Avg Sharpe:       1.26
  Avg Return:       20.8%
  Avg Max Drawdown: -10.4%
  Avg Calmar:       10.68
  Avg Trades/fold:  6
  Folds profitable: 6/8
  Sharpe OOS/IS:    0.57  (

ema adx 400/500: 1.13, 0.68
nwither 300: 1,21 , 0.73
neither 400/500: 1.06, 0.64
ema: 400 1.28 check     400 8 folds 1.36

---
## Granular Results and Parameter Stability

Per-fold IS vs OOS performance. Each row is one fold — compare `train_*` vs `test_*` columns to assess overfitting.

| Column | Description |
|---|---|
| `*_sharpe` `*_return` `*_drawdown` `*_calmar` | Core performance metrics |
| `*_trades` `*_winrate` `*_profit_factor` | Trade statistics |
| `optuna_score` | Best score achieved on training window |
| `param_*` | Best parameter values per fold e.g. `param_ema_span` |

**Concensus Parameters** - use to anchor: the engine determines stability using the coefficient of variation (CV) — the standard deviation of a parameter's best values across all folds divided by their median.

>CV < 0.15: indicates the strategy  relies on value rather than it being noise-fitted to a specific period — making it safe to fix for future runs. A high CV means the parameter is period-sensitive and should stay free.

In [120]:
# ── fold summary table ────────────────────────────────────────────────────────
display_cols = [
   # 'train_start', 'train_end',
    'test_start', 'test_end',
    'train_sharpe', 'test_sharpe',
    'train_return', 'test_return',
    'train_drawdown', 'test_drawdown',
    'test_trades',  'test_winrate', 
   # 'optuna_score',

]
display(results['results_df'][display_cols].round(2))

# ── parameter stability ───────────────────────────────────────────────────────
stability = results['stability_df'].copy()
stability['stable'] = stability['stable'].map({True: '✓', False: ''})
stability['fixed']  = stability['fixed'].map({True: '✓', False: ''})
stability = stability[['param', 'median', 'std', 'cv', 'stable', 'fixed']].round(2)
display(stability.sort_values('cv'))

# ── consensus params ──────────────────────────────────────────────────────────
stable = results['stability_df'][results['stability_df']['cv'] < 0.15]

print('Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:')
for _, row in stable.iterrows():
    v = results['consensus_params'][row['param']]
    v_fmt = int(round(v)) if isinstance(v, float) and v == int(v) else round(v, 4) if isinstance(v, float) else v
    print(f"    '{row['param']}': {v_fmt},")
    
print('\nConsensus parameters (median across folds):')
for k, v in results['consensus_params'].items():
    print(f'  {k:<30} = {round(v, 2) if isinstance(v, float) else v}')

,test_start,test_end,train_sharpe,test_sharpe,train_return,test_return,train_drawdown,test_drawdown,test_trades,test_winrate
0,2023-04-09,2023-08-23,2.20,0.43,5.20,0.02,-0.26,-0.05,2,0.50
1,2023-08-24,2024-01-07,2.43,2.69,5.41,0.55,-0.16,-0.09,13,0.69
2,2024-01-08,2024-05-23,2.24,4.24,4.24,0.63,-0.16,-0.06,5,0.80
3,2024-05-24,2024-10-07,2.40,-1.82,5.83,-0.14,-0.16,-0.14,3,0.00
4,2024-10-08,2025-02-21,2.16,0.86,4.30,0.09,-0.19,-0.12,7,0.57
5,2025-02-22,2025-07-08,2.12,1.53,5.19,0.19,-0.26,-0.14,5,0.60
6,2025-07-09,2025-11-22,2.08,2.71,5.24,0.37,-0.25,-0.12,7,0.71
7,2025-11-23,2026-04-08,2.13,-0.56,4.97,-0.04,-0.22,-0.13,2,0.00


,param,median,std,cv,stable,fixed
0,ema_span,28.00,0.00,0.00,✓,✓
10,stop_mult_pos_caution,0.50,0.00,0.00,✓,✓
9,max_leverage,2.60,0.00,0.00,✓,✓
16,obv_ma_period,20.00,0.00,0.00,✓,✓
8,risk_per_trade,0.04,0.00,0.00,✓,✓
1,swing_caution,5.00,0.00,0.00,✓,✓
17,obv_lookback,28.00,0.00,0.00,✓,✓
15,vol_ma_period,38.00,1.09,0.03,✓,
7,stop_atr_scale,1.36,0.10,0.07,✓,
3,atr_caution,21.00,1.56,0.07,✓,


Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:
    'ema_span': 28,
    'swing_caution': 5,
    'swing_stop': 3,
    'atr_caution': 21,
    'atr_stop': 28,
    'stop_atr_scale': 1.3585,
    'risk_per_trade': 0.045,
    'max_leverage': 2.6,
    'stop_mult_pos_caution': 0.5,
    'stop_mult_pos_normal': 1.4718,
    'vol_ma_period': 38,
    'obv_ma_period': 20,
    'obv_lookback': 28,

Consensus parameters (median across folds):
  ema_span                       = 28
  swing_caution                  = 5
  swing_stop                     = 3
  atr_caution                    = 21
  atr_stop                       = 28
  atr_size                       = 8
  adx_override                   = 57
  stop_atr_scale                 = 1.36
  risk_per_trade                 = 0.04
  max_leverage                   = 2.6
  stop_mult_pos_caution          = 0.5
  stop_mult_pos_normal           = 1.47
  stop_mult_ent_both             = 1.6
  stop_mult_ent_caution          = 0.38
  stop_mult_ent_normal 

---
## Parameter Robustness Checks

### Plateau Analysis
Sweep each free parameter across its range while holding others at consensus (median) value then evaluates the `score` at each point by backtesting over entire lookback.

The stability table (CV across folds) tells you *"does the optimizer consistently pick the same value?"*  

Plateau analysis tells you *"if that value were slightly wrong, would performance collapse?"*  

**Plateau %** - what fraction of each parameter's range stays within `threshold`% (default 20) of peak score: >60% = `robust plateau`, 30–60% = `moderate`, <30% = `fragile` (consider anchoring). `N/A` means every sweep point failed rejection filters — the strategy is completely intolerant of movement on that dimension.

>Run time: `n_free_params` × `n_steps`

### Perturbation test
Jitters all free parameters by ±5/10/20% of their range (50 random samples per offset range). Measures how much the score degrades vs the base

Test whether optimum is a broad hill in `#free params`-D space or a narrow spike

**>15%:** fragile optimum, consider reducing free parameters

In [112]:
# ── 1-D sensitivity sweeps around consensus params ─────────────────────────
sweep_results = plateau_analysis(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    n_steps      = 20, #Adjust for number of steps around concensus per parameter
)

# ── text verdicts ──────────────────────────────────────────────────────────
verdict_df = plateau_summary(
    sweep_results,
    base_params = results['consensus_params'],
    stability_df = results['stability_df'],  
    threshold   = 0.20, #Adjust for % around peak score
)

# ── neighbouahood perturbation ────────────────────────────────────────────
# Randomly jitters ALL free params simultaneously.
# If mean score degrades >15% at ±10% offset, the optimum is fragile.

perturb_df = perturbation_test(
    df           = df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = COST,
    pct_offsets  = (0.05, 0.10, 0.20),   # ±5%, ±10%, ±20% of range
    n_samples    = 50,                     # random perturbations per offset level
)


═══════════════════════════════════════════════════════════════════════════════════════════
PLATEAU ANALYSIS — PARAMETER ROBUSTNESS
═══════════════════════════════════════════════════════════════════════════════════════════
Parameter                 Consensus Peak Score  Plateau %  Fold CV Verdict                 
──────────────────────── ────────── ────────── ────────── ──────── ────────────────────────
adx_override                     50      0.870      100.0%    0.260 Robust                  
stop_mult_pos_normal         1.3820      0.826      100.0%    0.276 Robust                  
vol_ma_period                    38      0.823      100.0%    0.026 Robust                  
atr_size                          7      0.818      100.0%    0.267 Robust                  
atr_stop                         27      0.801      100.0%    0.088 Robust                  
stop_mult_ent_normal         0.6302      0.796      100.0%    0.511 Robust                  
stop_mult_ent_both           1.69

### 1-D sweep charts:
| Element | Meaning | Good | Bad |
|---------|---------|------|-----|
| **Blue curve** | Composite score at each value of the parameter, with all others held at consensus | Flat-topped curve — performance is insensitive to the exact value | Narrow spike — optimizer latched onto one specific value, everything nearby is worse |
| **Red dashed line** | Where the consensus value sits | On the flat top of the curve | On a steep slope or at the edge of a cliff |
| **Green dashed line** | Cutoff at 80% of peak score — the boundary between plateau and non-plateau | Blue curve stays above this line across most of the range | Blue curve dips below it quickly either side of the peak |
| **Green shading** | Plateau region — all values where the score stays within 20% of the peak | Wide green band spanning most of the range (robust) | Thin sliver or no shading at all (fragile/overfit) |

 If concensus on steep slope: parameter **REGIME SENSITIVE** - do not fix, backtests are disagreeing, want to fix parameters on flat top.

In [113]:

# ── visual sweep curves ───────────────────────────────────────────────────
plot_plateau_analysis(
    sweep_results    = sweep_results,
    consensus_params = results['consensus_params'],
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    threshold        = 0.20,
    show             = False,
    save_html        = None,
)

---
## Results Charts and Cost Stress Test

| Parameter | Description | Default |
|---|---|---|
| `show_fold_perf` | IS vs OOS bars for return, Sharpe, drawdown per fold | `False` |
| `show_param_evol` | Parameter evolution across folds with ±1 std bands | `False` |
| `show_oos_equity` | Combined OOS equity curve + drawdown with fold boundaries | `True` |
| `show_trades` | Overlay entry/exit markers on OOS equity chart | `False` |
| `benchmark_data` | DataFrame with `Close` column for buy & hold comparison | `None` |
| `save_html_dir` | Directory path to save charts as HTML files, or `None` | `None` |

**Cost Stress Test:** re-run the combined OOS backtest at 1×, 1.5×, 2×, 3× the base cost. Fragile strategies collapse; robust ones degrade gradually.

In [118]:
plot_walk_forward_results(
    results          = results,
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    benchmark_data   = df,
    show             = True,
    save_html_dir    = None,
    show_fold_perf   = False,   # IS vs OOS bars by fold
    show_param_evol  = False,   # parameter evolution across folds
    show_oos_equity  = True,   # combined OOS equity curve
    show_trades      = False,  # trade markers on OOS equity chart
)

# ── transaction cost stress test ──────────────────────────────────────────

if results['oos_combined_df'] is not None:
    cost_df = cost_stress_test(
        oos_combined_df  = results['oos_combined_df'],
        cost_multipliers = (1.0, 1.5, 2.0, 3.0),
        base_cost        = COST,
    )
else:
    print('No combined OOS dataframe — skip cost stress test')


═══════════════════════════════════════════════════════════════════════════
TRANSACTION COST STRESS TEST
═══════════════════════════════════════════════════════════════════════════
    Cost   Mult   Sharpe     Return      MaxDD   Calmar       PF
──────── ────── ──────── ────────── ────────── ──────── ────────
  0.0010   1.0x     1.57    272.38%    -24.29%     2.26     3.33
  0.0015   1.5x     1.52    255.86%    -24.82%     2.12     3.33
  0.0020   2.0x     1.48    240.07%    -25.36%     1.98     3.33
  0.0030   3.0x     1.38    210.54%    -26.41%     1.74     3.33


In [119]:
results['oos_combined_df'].to_pickle('eth_oos.pkl')